# Batch processing template / 批次分析模板

這份 notebook 示範如何把 napari-assistant 建立的單張影像流程，轉成多張影像的批次分析。

**Teaching idea / 教學重點**
 
  先在 napari 中完成單張影像流程。 
  再把處理步驟貼到下方的迴圈中。  
  最後輸出合併後的 CSV 結果表。




## 1. Import packages / 匯入套件

  
這些套件用來讀圖、分割、以及輸出量測結果。

**Main packages / 主要套件**
- `Path`: handle file paths / 處理路徑
- `imread`: read `.tif` images / 讀取 `.tif`
- `pyclesperanto`: GPU-style image processing / 影像處理
- `napari_simpleitk_image_processing`: label processing / label 處理
- `regionprops_table`: measurement / 量測
- `pandas`: save results as table / 表格整理


In [ ]:
from pathlib import Path
from skimage.io import imread
import pyclesperanto as cle
import napari_simpleitk_image_processing as nsitk
import napari_segment_blobs_and_things_with_membranes as nsbatwm
import pandas as pd
import numpy as np
from skimage.measure import regionprops_table
import napari

# 初始化 Viewer (可選)
# viewer = napari.Viewer()


## 2. Set input folder / 設定輸入資料夾

 
請把要分析的 `.tif` 影像放在 `data/batch` 資料夾中。

這個 notebook 會自動讀取資料夾中的所有 `.tif` 檔案。


In [ ]:
# 請確保你的 ipynb 路徑中有一個名為 batch 的資料夾，裡面放著 .tif 檔案
input_dir = Path("data/batch image2")
results_list = [] 

# 檢查資料夾是否存在
if not input_dir.exists():
    print(f"找不到資料夾：{input_dir.absolute()}，請先建立它並放入圖片！")

## 3. Main batch loop / 主要批次迴圈



1. Read one image / 讀取單張影像  
2. Run the same workflow / 執行相同的處理流程  
3. Measure objects / 量測物件  
4. Append the result / 加到結果表中  

### Important notes / 重要提醒
  貼上 assistant 程式碼時，要保留 `for` 迴圈內的縮排。
 
  每個人從 assistant 產生的變數名稱可能不同。
  為了避免錯誤，請把最後的 label 結果指定給 `final_label_image`。


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
# Use glob to find all .tif files / 使用 glob 抓取所有 .tif 檔案
for img_path in input_dir.glob("*.tif"):
    image_data = imread(img_path)

    # 4. Image processing workflow / 影像處理流程
    # Paste the code generated by napari-assistant below. 請把 napari-assistant 產生的程式碼貼在這裡。
    # Keep the indentation (4 spaces) inside the loop. 注意：要維持 for 迴圈內的縮排（4 個空格） 



    

    # ===== Assistant workflow ends here / Assistant 流程到此結束 =====
    
    
    # IMPORTANT: assign your final label result to final_label_image
    # 重要：請把「最後的 label 結果」指定給 final_label_image
    final_label_image = image6_eloe   # change this line if your final variable name is different
    # 如果你最後的變數不是 image6_eloe，請改這一行

    # 5. Measurement / 量測
    # regionprops_table needs NumPy arrays.
    # regionprops_table 需要 NumPy array，因此先做轉換。
    label_np = np.asarray(final_label_image).astype(np.int32)
    intensity_np = np.asarray(image_data).astype(np.float32)

    props = regionprops_table(
        label_np,
        intensity_image=intensity_np,
        properties=['label', 'area', 'mean_intensity']
    )

    # Convert to DataFrame and add file name / 轉成表格並加上檔名
    df_single_image = pd.DataFrame(props)
    df_single_image['filename'] = img_path.name

    # Count objects / 計算物件數
    current_count = len(df_single_image)

    # Store result for this image / 儲存這張圖的結果
    results_list.append(df_single_image)
    print(f"完成: {img_path.name} -> 細胞數: {current_count}")

# 6. Merge all results and export CSV / 合併所有結果並輸出 CSV
if results_list:
    final_report = pd.concat(results_list, ignore_index=True)
    final_report = final_report[final_report['label'] > 0]
    cols = ['filename'] + [c for c in final_report.columns if c != 'filename'] #filename 在第一欄
    final_report = final_report[cols]
    final_report.to_csv("final_analysis.csv", index=False, encoding="utf_8_sig")
    print(f"\n🎉 任務完成！表格已存入 final_analysis.csv")
else:
    print("⚠️ No results were generated. Please check your folder path and workflow.")


## 4. Output / 輸出結果
  
當迴圈結束後，所有量測結果會合併成一份 `final_analysis.csv`。



